# VIS4AI · Notebook 01 — Setup and first run

Goal of this notebook: get everything working and **let both models solve the same GSM8K math problems**.
No metrics or plots yet — just a clean setup and the raw answers, saved to Google Drive.

| Model | Role |
|---|---|
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | main model: a small *reasoning* model that "thinks" in `<think> … </think>` before answering |
| `Qwen/Qwen2.5-Math-1.5B` | baseline: the math base model that R1-Distill was fine-tuned from (no long reasoning) |

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (allow Google Drive access when asked)
3. **Disconnected?** Set `RESUME_RUN` in the settings to the name of the run folder and *Run all* again. Answers are saved after every batch, and problems that are already done are skipped.

## 1. Setup
Install packages, get the repo's `utils` helpers, check the GPU and connect Google Drive (where all results are stored).

*You may see a warning "You are sending unauthenticated requests to the HF Hub". It is harmless: the models and data are public.*

In [1]:
!pip install -q "transformers>=4.56" "accelerate>=0.33" "datasets>=2.20"

# utils/ lives in the repo root: clone the repo in Colab, use the parent folder when run locally from notebooks/
import os, sys
if os.path.exists("/content"):
    if not os.path.exists("/content/AI-VisualizationProject-TheFinalGroup"):
        !git clone -q https://github.com/lucamcwood/AI-VisualizationProject-TheFinalGroup.git /content/AI-VisualizationProject-TheFinalGroup
    sys.path.insert(0, "/content/AI-VisualizationProject-TheFinalGroup")
else:
    sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import pandas as pd
import torch
from datasets import load_dataset
import transformers

from utils.llm_helpers import (answer_cell, extract_answer, is_correct, load_results,
                               make_run_dir, run_model, to_number, write_run_summary)

assert torch.cuda.is_available(), "No GPU! Go to Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

GPU: Tesla T4


### Settings
All settings are in this one cell. Each model is run the way its authors recommend, so the settings differ per model:

| | R1-Distill | Qwen2.5-Math |
|---|---|---|
| prompt | the question + instruction as one user message in the model's chat template, no system prompt (DeepSeek) | the **same text, but as plain text without a chat template** (`chat_template=False`). It is a base model and never learned the chat format: with it, it wrote junk tokens where the chat turn should end and often looped. Base Qwen2.5-Math is known to work best with no template (Liu et al., 2025, *Understanding R1-Zero-Like Training*) |
| decoding | sampling, temperature 0.6, top-p 0.95 (DeepSeek; greedy can make it loop) | greedy (Qwen) |
| `max_new_tokens` | 4096 — its thinking is long; with 2048, 1 in 5 answers was cut off | 1024 — answers are short |
| `stop_after_answer` | no — the model ends by itself after `</think>` + answer | **yes** — as a base model it does not stop by itself and starts inventing new questions after its answer, so we stop it as soon as it has written a `\boxed{...}` answer |
| answer = | the last `\boxed{...}` **after** `</think>` (`thinks=True`). No `</think>` = still thinking when the token limit hit = no answer | the first non-empty `\boxed{...}` (later boxes would belong to invented questions) |

* `N_PROBLEMS`: how many GSM8K problems to try. Raising it later only runs the new problems.
* `RESUME_RUN`: `None` starts a fresh run in a new folder `run_<i>_<date>_<time>`, so old and new answers are never mixed. **To resume after a disconnect, set it to that folder's name** (e.g. `"run_3_2026-11-08_14-30"`).
* All settings are written to `parameters.txt` in the run folder.

In [ ]:
MODELS = {
    "r1-distill": dict(name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", thinks=True,
                       chat_template=True,  sampling=True,  max_new_tokens=4096, stop_after_answer=False),
    "qwen-math":  dict(name="Qwen/Qwen2.5-Math-1.5B", thinks=False,
                       chat_template=False, sampling=False, max_new_tokens=1024, stop_after_answer=True),
}

N_PROBLEMS  = 50       # number of GSM8K test problems
BATCH_SIZE  = 16       # problems generated at the same time (16 fits on a T4)
TEMPERATURE = 0.6      # only used by models with sampling=True
TOP_P       = 0.95
SEED        = 42

INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

RESUME_RUN  = None     # None = new run; set to a run folder name (e.g. "run_3_2026-11-08_14-30") to resume it
RESULTS_DIR = make_run_dir("/content/drive/MyDrive/VIS4AI/results", RESUME_RUN)
write_run_summary(RESULTS_DIR, models=MODELS, n_problems=N_PROBLEMS, batch_size=BATCH_SIZE,
                  temperature=TEMPERATURE, top_p=TOP_P, seed=SEED, instruction=INSTRUCTION,
                  gpu=torch.cuda.get_device_name(0), torch=torch.__version__, transformers=transformers.__version__)
print("Results are saved in:", RESULTS_DIR)

Results are saved in: /content/drive/MyDrive/VIS4AI/results/run1


## 2. Load the dataset
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) test split: 1,319 grade-school math word problems. The correct answer is the number after `####` in the solution text.

In [5]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

problems = [
    {"id": i,
     "question": gsm8k[i]["question"],
     "gold": gsm8k[i]["answer"].split("####")[-1].strip().replace(",", "")}
    for i in range(N_PROBLEMS)
]

print(f"{len(problems)} problems loaded. Example:\n")
print(problems[0]["question"])
print("\nCorrect answer:", problems[0]["gold"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

50 problems loaded. Example:

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Correct answer: 18


## 3. Helper functions
They are in [`utils/llm_helpers.py`](../utils/llm_helpers.py):
* **`build_prompt`** — the question plus *"Please reason step by step, and put your final answer within \boxed{}."* With `chat_template=True` the tokenizer's chat template wraps it in the model-specific special tokens; otherwise it is plain text.
* **`extract_answer`** — gets the answer out of the output, following the "answer =" row of the settings table. Empty `\boxed{}` (a repeated instruction) is ignored. `None` means no answer.
* **`load_results` / `save_results`** — read/append a `.jsonl` file on Drive (one line per problem), which makes the run resumable.
* **`load_model` / `free_gpu`** — load a model in float16 (the T4 has no bfloat16) and free the GPU afterwards.
* **`generate_answers`** — let the model answer a batch of questions. Also records how many tokens it wrote and whether it **finished** (`False` = it hit the token limit and was cut off).
* **`run_model`** — runs one model on all problems that are not done yet, saving after every batch.

In [ ]:
# Quick sanity check of the answer parsing
assert extract_answer("thinking... </think> so \\boxed{18}.", thinks=True) == "18"
assert extract_answer("thinking \\boxed{5} but ran out of tokens", thinks=True) is None
assert extract_answer("\\boxed{7}. Please put your answer within \\boxed{}. \\boxed{3}", thinks=False) == "7"
assert extract_answer("no box here", thinks=False) is None
assert is_correct("\\$1,234", "1234") and is_correct("20\\%", "20") and is_correct("75.00", "75")
assert to_number("\\frac{3}{4}") == 0.75 and not is_correct(None, "5")
print("Answer parsing works.")

Answer parsing works.


## 4. Run model 1: DeepSeek-R1-Distill-Qwen-1.5B
The reasoning model writes long "thinking" before answering. A batch takes as long as its longest answer, so expect roughly 5 minutes per batch of 16 (about 20 minutes for 50 problems).

In [9]:
run_model("r1-distill", MODELS["r1-distill"], problems, RESULTS_DIR, INSTRUCTION, BATCH_SIZE, SEED, TEMPERATURE, TOP_P)

r1-distill: 50 problems already done, 0 to go. Saving to /content/drive/MyDrive/VIS4AI/results/run1/r1-distill.jsonl
Done: 43/50 correct | 5 cut off at the token limit | 5 without a boxed answer


## 5. Run model 2: Qwen2.5-Math-1.5B
Same problems. The first model was removed from the GPU, so there is room for this one. This model is much faster (short answers, stopped right after its `\boxed{}` answer).

In [10]:
run_model("qwen-math", MODELS["qwen-math"], problems, RESULTS_DIR, INSTRUCTION, BATCH_SIZE, SEED, TEMPERATURE, TOP_P)

qwen-math: 0 problems already done, 50 to go. Saving to /content/drive/MyDrive/VIS4AI/results/run1/qwen-math.jsonl


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-Math-1.5B | GPU memory used: 3.1 GB
  saved 16/50  (33 s)
  saved 32/50  (53 s)
  saved 48/50  (54 s)
  saved 50/50  (15 s)
Done: 40/50 correct | 2 cut off at the token limit | 2 without a boxed answer


## 6. Look at the answers
Both models side by side, one row per problem (read back from Drive, so this also works after a disconnect).

`18 ✓` = correct, `180 ✗` = wrong, `cut off` = hit the token limit before answering, `no answer` = finished without a `\boxed{}` answer.
The answers are re-extracted from the saved outputs here, so improving `extract_answer` never requires re-running the models.

In [ ]:
comparison = pd.DataFrame(problems).set_index("id")[["gold"]]
for key, cfg in MODELS.items():
    results = {r["id"]: r for r in load_results(RESULTS_DIR / f"{key}.jsonl")}
    comparison[key] = [answer_cell(results[i], cfg["thinks"]) if i in results else "not run"
                       for i in comparison.index]

pd.set_option("display.max_rows", None)
comparison

,gold,r1-distill,qwen-math
id,,,
0,18,18 ✓,18 ✓
1,3,3 ✓,3 ✓
2,70000,70000 ✓,70000 ✓
3,540,180 ✗,180 ✗
4,20,20 ✓,140 ✗
5,64,64 ✓,64 ✓
6,260,260 ✓,260 ✓
7,160,cut off,140 ✗
8,45,45 ✓,315 ✗


In [12]:
# Read one full output. Change these two values to look at other problems / the other model.
MODEL_KEY  = "r1-distill"    # "r1-distill" or "qwen-math"
PROBLEM_ID = 0

example = {r["id"]: r for r in load_results(RESULTS_DIR / f"{MODEL_KEY}.jsonl")}[PROBLEM_ID]
print("QUESTION:", example["question"])
print(f"\nCORRECT ANSWER: {example['gold']} | "
      f"MODEL ANSWER: {extract_answer(example['output'], MODELS[MODEL_KEY]['thinks'])} | "
      f"{example['n_tokens']} tokens | finished: {example['finished']}")
print("\nMODEL OUTPUT:\n", example["output"])

QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

CORRECT ANSWER: 18 | MODEL ANSWER: 18 | 3381 tokens | finished: True

MODEL OUTPUT:
 Okay, so I need to figure out how much money Janet makes every day at the farmers' market. Let me break down the problem step by step to make sure I understand everything correctly.

First, Janet's ducks lay 16 eggs per day. That's the total number of eggs available each day. Now, she has some consumption and some production involved, so I need to calculate how many eggs she actually sells at the market.

She eats three eggs for breakfast every morning. Since a day has 24 hours, the number of mornings in a day is 24. So, if she eats three eggs each morning, I can calculate the total number of eggs she consumes for b